# Experiment 2  CatBoost vs Gradient Boosting



### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib


In [2]:
dt = joblib.load("starting_split.joblib")
x_train = dt["x_train"]
y_train = dt["y_train"]


In [3]:
x_train.shape


(4507, 23)

In [4]:
y_train.value_counts()


diabetes
0.0    3685
1.0     822
Name: count, dtype: int64

In [5]:
from pathlib import Path

OUTPUT = Path("results/experiment2_catboost_vs_gradient_boosting")
OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUTPUT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


### MICE imputation

In [6]:
# MICE data for each fold.
from pathlib import Path

def apply_mice(train_data, validation_data, folder_name):
    folder = Path(folder_name)
    frames = []
    for filename, original in [('train_imputed.csv', train_data),('validation_imputed.csv', validation_data)]:
        path = folder / filename
        if not path.exists():
            raise FileNotFoundError(f'Missing {path}')
        frame = pd.read_csv(path)
        if frame.isna().any().any():
            raise ValueError(f'Missing values in baseline MICE data: {path}')
        frame.index = original.index
        frames.append(frame)
    return tuple(frames)


### Prepare five folds with OneHotEncoder and RFE20

In [7]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [9]:
# onehot encoding and RFE20 selection in each fold.

prep_folds = []
sel_feats = []

for fold_num, (train_idx, val_idx) in enumerate(cv.split(x_train, y_train), start=1):
    print("Preparing fold", fold_num, "/5")
    x_tr = x_train.iloc[train_idx]
    x_val = x_train.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    x_tr_imp, x_val_imp = apply_mice( x_tr, x_val, f"mice_cache/fold_{fold_num}" )

    encoder = OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False)
    x_tr_enc = encoder.fit_transform(x_tr_imp.astype(str))
    x_val_enc = encoder.transform(x_val_imp.astype(str))
    print("Training shape after OneHotEncoder:", x_tr_enc.shape)
    print("Validation shape after OneHotEncoder:", x_val_enc.shape)

    sel_lr = LogisticRegression(solver="liblinear", max_iter=1000, random_state=42)
    selector = RFE(estimator=sel_lr, n_features_to_select=20, step=1)
    x_tr_sel = selector.fit_transform(x_tr_enc, y_tr)
    x_val_sel = selector.transform(x_val_enc)
    print("Training shape after RFE20:", x_tr_sel.shape)

    feature_names = encoder.get_feature_names_out()[selector.support_]
    for feature_name in feature_names:
        sel_feats.append({"fold": fold_num, "feature": feature_name})

    prep_folds.append({ "fold": fold_num, "encoder": encoder, "selector": selector, "x_train": x_tr_sel, "y_train": y_tr.to_numpy(),
        "x_val": x_val_sel, "y_val": y_val.to_numpy(),"features": x_tr_sel.shape[1]})




Preparing fold 1 /5
Training shape after OneHotEncoder: (3605, 68)
Validation shape after OneHotEncoder: (902, 68)
Training shape after RFE20: (3605, 20)
Preparing fold 2 /5
Training shape after OneHotEncoder: (3605, 68)
Validation shape after OneHotEncoder: (902, 68)
Training shape after RFE20: (3605, 20)
Preparing fold 3 /5
Training shape after OneHotEncoder: (3606, 68)
Validation shape after OneHotEncoder: (901, 68)
Training shape after RFE20: (3606, 20)
Preparing fold 4 /5
Training shape after OneHotEncoder: (3606, 68)
Validation shape after OneHotEncoder: (901, 68)
Training shape after RFE20: (3606, 20)
Preparing fold 5 /5
Training shape after OneHotEncoder: (3606, 68)
Validation shape after OneHotEncoder: (901, 68)
Training shape after RFE20: (3606, 20)


In [10]:
pd.DataFrame(sel_feats).to_csv(OUTPUT / "rfe20_selected_features.csv", index=False)

In [11]:
print("RFE20 ready:", len(prep_folds), "folds")


RFE20 ready: 5 folds


### Apply SMOTE to training data only

In [12]:
# Apply SMOTE
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
from imblearn.over_sampling import SMOTE

smote_folds = []

for fold in prep_folds:
    smote = SMOTE(random_state=42)
    x_bal, y_bal = smote.fit_resample(fold["x_train"], fold["y_train"])
    smote_folds.append({ "fold": fold["fold"], "encoder": fold["encoder"], "selector": fold["selector"],
        "x_train": x_bal, "y_train": y_bal,"x_val": fold["x_val"], "y_val": fold["y_val"], "features": fold["features"] })


c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 546, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1022, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  

In [13]:
print("SMOTE ready:", len(smote_folds), "folds")


SMOTE ready: 5 folds


# Model Training




## Load baseline Gradient Boosting results

In [14]:
# Load the baseline Gradient Boosting scores.
from sklearn.metrics import ( accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc, balanced_accuracy_score)
from IPython.display import display

met_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "balanced_accuracy", "specificity"]

base_dir = Path("results/baseline/gradient_boosting")
gb_met = pd.read_csv(base_dir / "cv_fold_metrics.csv")
gb_met = gb_met.rename(columns={ "Fold": "fold", "Accuracy": "accuracy", "Precision": "precision", "Recall": "recall", "F1-score": "f1", "AUROC": "roc_auc", "PR-AUC": "pr_auc"})
gb_met = gb_met.sort_values("fold").reset_index(drop=True)
assert gb_met["fold"].tolist() == [1, 2, 3, 4, 5]

for i, fold in enumerate(prep_folds):
    y_true = fold["y_val"]
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    tp = gb_met.loc[i, "recall"] * n_pos
    n_correct = gb_met.loc[i, "accuracy"] * len(y_true)
    specificity = (n_correct - tp) / n_neg
    gb_met.loc[i, "specificity"] = specificity
    gb_met.loc[i, "balanced_accuracy"] = (gb_met.loc[i, "recall"] + specificity) / 2

gb_met["features"] = 20
gb_met = gb_met[["fold", "features"] + met_cols]


In [15]:
# Find the mean scores across the five folds.
gb_acc = gb_met["accuracy"].mean()
gb_prec = gb_met["precision"].mean()
gb_rec = gb_met["recall"].mean()
gb_f1 = gb_met["f1"].mean()
gb_roc_auc = gb_met["roc_auc"].mean()
gb_pr_auc = gb_met["pr_auc"].mean()
gb_bal_acc = gb_met["balanced_accuracy"].mean()
gb_spec = gb_met["specificity"].mean()

gb_sum = gb_met[met_cols].agg(["mean", "std"]).T


In [16]:
display(gb_met)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1,20,0.745011,0.378277,0.612121,0.467593,0.774220,0.401604,0.693442,0.774763
1,2,20,0.738359,0.373665,0.636364,0.470852,0.758423,0.433218,0.698779,0.761194
2,3,20,0.753607,0.393382,0.652439,0.490826,0.795124,0.509273,0.714279,0.776119
3,4,20,0.720311,0.351351,0.634146,0.452174,0.758774,0.446217,0.686815,0.739484
4,5,20,0.764706,0.404762,0.621951,0.490385,0.784662,0.451549,0.709212,0.796472


In [17]:
display(gb_sum)


,mean,std
accuracy,0.744399,0.016690
precision,0.380288,0.020345
recall,0.631404,0.015295
f1,0.474366,0.016418
roc_auc,0.774240,0.016079
pr_auc,0.448372,0.039181
balanced_accuracy,0.700505,0.011245
specificity,0.769607,0.021029


In [18]:
gb_met.to_csv(OUTPUT / "baseline_gb_fold_metrics.csv", index=False)
gb_sum.to_csv(OUTPUT / "baseline_gb_summary.csv")
print("Loaded baseline Gradient Boosting results; no Gradient Boosting training is performed.")


Loaded baseline Gradient Boosting results; no Gradient Boosting training is performed.


## CatBoost

In [19]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Importing catboost

In [20]:
import sys
if Path(".experiment_dependencies/catboost").is_dir():
    sys.path.insert(0, str(Path(".experiment_dependencies").resolve()))
from catboost import CatBoostClassifier


### Hyperparameter tuning using five folds

In [21]:
import json
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid

In [22]:
# CatBoost parameter grid.
cat_dir = OUTPUT / "catboost_108_grid"
cat_dir.mkdir(parents=True, exist_ok=True)
cat_tune_file = cat_dir / "tuning_results.json"
cat_model_file = MODEL_DIR / "cat_108_grid.joblib"

cat_est = CatBoostClassifier(random_seed=42, loss_function="Logloss", verbose=False,allow_writing_files=False, bootstrap_type="Bernoulli", sampling_frequency="PerTree")
cat_grid = {"iterations": [100, 200, 300],"learning_rate": [0.01, 0.05, 0.1], "depth": [3, 5, 7],
    "subsample": [0.8, 1.0],"l2_leaf_reg": [3, 5]}

In [23]:
# Load saved settings
if cat_model_file.exists():
    saved_cat = joblib.load(cat_model_file)

    if isinstance(saved_cat, dict):
        cat_params = saved_cat.get("best_parameters")
        if cat_params is None:
            saved_params = saved_cat["fold_models"][0]["model"].get_params()
            cat_params = {key: saved_params[key] for key in cat_grid}
    else:
        saved_params = saved_cat.get_params()
        cat_params = {key: saved_params[key] for key in cat_grid}
    cat_res = {"Model": "CatBoost", "Best Parameters": cat_params}
    print("Saved CatBoost model found; skipping tuning.")
elif cat_tune_file.exists():
    cat_res = json.loads(cat_tune_file.read_text(encoding="utf-8"))
    print("Saved CatBoost tuning results found; skipping tuning.")
else:

    param_sets = list(ParameterGrid(cat_grid))
    best_f1 = -1
    tune_hist = []

    for number, settings in enumerate(param_sets, start=1):
        print("CatBoost parameter combination", number, "/", len(param_sets), settings)
        f1_scores = []

        for fold in smote_folds:
            cat_model = clone(cat_est)
            cat_model.set_params(**settings)
            cat_model.fit(fold["x_train"], fold["y_train"])

            predictions = cat_model.predict(fold["x_val"]).ravel()
            fold_f1 = f1_score(fold["y_val"], predictions, zero_division=0)
            f1_scores.append(fold_f1)

        mean_f1 = float(np.mean(f1_scores))
        std_f1 = float(np.std(f1_scores, ddof=1))
        tune_hist.append({"Model": "CatBoost", "Parameters": settings,"Mean F1": mean_f1, "F1 Std": std_f1})

        if mean_f1 > best_f1:
            best_f1 = mean_f1
            best_params = settings


    cat_res = {"Model": "CatBoost","Best Parameters": best_params,"Best CV F1": best_f1,"Search History": tune_hist}
    cat_tune_file.write_text(json.dumps(cat_res, indent=2), encoding="utf-8")
    pd.DataFrame(tune_hist).to_csv(cat_dir / "tuning_history.csv", index=False)


Saved CatBoost model found; skipping tuning.


In [24]:
cat_params = cat_res["Best Parameters"]
print("CatBoost best parameters:", cat_params)
if "Best CV F1" in cat_res:
    print("Best mean CV F1:", cat_res["Best CV F1"])


CatBoost best parameters: {'depth': 5, 'iterations': 200, 'l2_leaf_reg': 3, 'learning_rate': 0.01, 'subsample': 1.0}


### Train the model

In [25]:
# Train CatBoost -each fold
cat_models = []
cat_preds = []
cat_probs = []

for fold_num, fold in enumerate(smote_folds, start=1):
    print("CatBoost: training fold", fold_num, "/5")
    model = clone(cat_est).set_params(**cat_params)
    model.fit(fold["x_train"], fold["y_train"])

    y_pred = model.predict(fold["x_val"]).ravel()
    y_prob = model.predict_proba(fold["x_val"])[:, 1]
    cat_preds.append(y_pred)
    cat_probs.append(y_prob)

    cat_models.append({"fold": fold_num, "model": model, "encoder": fold["encoder"], "selector": fold["selector"], "input_columns": list(x_train.columns)})

CatBoost: training fold 1 /5
CatBoost: training fold 2 /5
CatBoost: training fold 3 /5
CatBoost: training fold 4 /5
CatBoost: training fold 5 /5


### Calculate validation metrics

In [26]:
cat_met = pd.DataFrame(columns=["fold", "features"] + met_cols)

for i, fold in enumerate(smote_folds):
    y_true = fold["y_val"]
    y_pred = cat_preds[i]
    y_prob = cat_probs[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)

    cat_fold_acc = accuracy_score(y_true, y_pred)
    cat_fold_prec = precision_score(y_true, y_pred, zero_division=0)
    cat_fold_rec = recall_score(y_true, y_pred)
    cat_fold_f1 = f1_score(y_true, y_pred)
    cat_fold_roc_auc = roc_auc_score(y_true, y_prob)
    cat_fold_pr_auc = auc(recall, precision)
    cat_fold_bal_acc = balanced_accuracy_score(y_true, y_pred)
    cat_fold_spec = recall_score(y_true, y_pred, pos_label=0)

    cat_met.loc[i] = [i + 1, fold["features"], cat_fold_acc, cat_fold_prec, cat_fold_rec, cat_fold_f1, cat_fold_roc_auc,
        cat_fold_pr_auc, cat_fold_bal_acc, cat_fold_spec]


### Calculate the mean and standard deviation across five folds

In [27]:
cat_acc = cat_met["accuracy"].mean()
cat_prec = cat_met["precision"].mean()
cat_rec = cat_met["recall"].mean()
cat_f1 = cat_met["f1"].mean()
cat_roc_auc = cat_met["roc_auc"].mean()
cat_pr_auc = cat_met["pr_auc"].mean()
cat_bal_acc = cat_met["balanced_accuracy"].mean()
cat_spec = cat_met["specificity"].mean()
cat_sum = cat_met[met_cols].agg(["mean", "std"]).T

In [28]:
display(cat_met)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,20.0,0.742794,0.383275,0.666667,0.486726,0.779088,0.404859,0.713252,0.759837
1,2.0,20.0,0.733925,0.373737,0.672727,0.480519,0.761054,0.430726,0.710176,0.747626
2,3.0,20.0,0.731410,0.370861,0.682927,0.480687,0.798495,0.506656,0.712562,0.742198
3,4.0,20.0,0.693674,0.332335,0.676829,0.445783,0.752610,0.438592,0.687126,0.697422
4,5.0,20.0,0.750277,0.395904,0.707317,0.507659,0.789957,0.470959,0.733577,0.759837


In [29]:
display(cat_sum)


,mean,std
accuracy,0.730416,0.021857
precision,0.371223,0.023839
recall,0.681293,0.015709
f1,0.480275,0.022256
roc_auc,0.776241,0.019239
pr_auc,0.450358,0.039331
balanced_accuracy,0.711339,0.016487
specificity,0.741384,0.025755


### Save the fold models and results

In [30]:
# Save the CatBoost models and scores.
cat_met.to_csv(OUTPUT / "cat_fold_metrics.csv", index=False)
cat_sum.to_csv(OUTPUT / "cat_summary.csv")
joblib.dump({"variant": "cat", "positive_class": 1, "best_parameters": cat_params, "requires_mice_imputed_input": True, "fold_models": cat_models}, cat_model_file)
print("Saved CatBoost fold models and results")

Saved CatBoost fold models and results


## Compare models

In [31]:
comp = pd.DataFrame({"variant": ['Baseline Gradient Boosting', 'CatBoost'],"accuracy": [gb_acc, cat_acc], "precision": [gb_prec, cat_prec],
    "recall": [gb_rec, cat_rec],"f1": [gb_f1, cat_f1], "roc_auc": [gb_roc_auc, cat_roc_auc], "pr_auc": [gb_pr_auc, cat_pr_auc],
"balanced_accuracy": [gb_bal_acc, cat_bal_acc], "specificity": [gb_spec, cat_spec],}).set_index("variant")

comp.to_csv(OUTPUT / "comparison_mean.csv")
display((comp * 100).style.format("{:.2f}%"))


,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
variant,,,,,,,,
Baseline Gradient Boosting,74.44%,38.03%,63.14%,47.44%,77.42%,44.84%,70.05%,76.96%
CatBoost,73.04%,37.12%,68.13%,48.03%,77.62%,45.04%,71.13%,74.14%


In [32]:
(comp * 100).to_csv(OUTPUT / "comparison_mean_percent.csv")

## Model Evaluation

Looking at the validation results of Both models, the baseline Gradient boosting model show
higher performance in accuracy, precision and balance accuracy. But since accuracy itself is not
enough to judge a model Catboost managed to score higher in recall , F1 score, PR-AUC and
AUROC compare to gradient boosting. What is most important in CatBoost is that its recall for
predicting the diabetes class is 5% higher than Gradient Boosting. We cannot decide that
CatBoost is the better model base on these results but it shows that this alternative approach is
useful for improving diabetes prediction even though it has slight lower accuracy and precision
score.


## Compare the models on the test set

I used the same 1,127 test respondents for both models. The saved Gradient Boosting model was trained on the full 80% training set after its settings were selected using five-fold cross-validation. CatBoost is trained on the same training data using its best settings from five-fold cross-validation. The test data is only transformed and evaluated.


In [33]:
# Use the same test split and preprocessing as the baseline.
x_test = dt["x_test"]
y_test = dt["y_test"]
x_train_imp, x_test_imp = apply_mice(x_train, x_test, "mice_cache/final")
preprocessing = joblib.load("models/baseline/preprocessing.joblib")
final_encoder = preprocessing["encoder"]
final_rfe = preprocessing["rfe"]



In [34]:
x_train_encoded = final_encoder.transform(x_train_imp.astype(str))
x_test_encoded = final_encoder.transform(x_test_imp.astype(str))
x_train_selected = final_rfe.transform(x_train_encoded)
x_test_selected = final_rfe.transform(x_test_encoded)

In [35]:
# SMOTE for training data
x_train_balanced, y_train_balanced = SMOTE(random_state=42).fit_resample(x_train_selected, y_train)
final_cat = clone(cat_est).set_params(**cat_params)

In [36]:
final_cat.fit(x_train_balanced, y_train_balanced)

CatBoostClassifier(allow_writing_files=False, bootstrap_type='Bernoulli', depth=5, iterations=200, l2_leaf_reg=3, learning_rate=0.01, loss_function='Logloss', random_seed=42, sampling_frequency='PerTree', subsample=1.0, verbose=False)

In [37]:
final_gb = joblib.load("models/baseline/gradient_boosting.joblib")

In [38]:
test_models = {"Gradient Boosting": final_gb, "CatBoost": final_cat}
test_scores = []
for model_name, trained_model in test_models.items():
    test_prediction = trained_model.predict(x_test_selected).ravel()
    test_probability = trained_model.predict_proba(x_test_selected)[:, 1]
    precision_values, recall_values, _ = precision_recall_curve(y_test, test_probability)
    test_scores.append({"Model": model_name, "Accuracy": accuracy_score(y_test, test_prediction),
        "Precision": precision_score(y_test, test_prediction, zero_division=0), "Diabetes recall": recall_score(y_test, test_prediction),
        "F1-score": f1_score(y_test, test_prediction), "AUROC": roc_auc_score(y_test, test_probability),
        "PR-AUC": auc(recall_values, precision_values), "Balanced accuracy": balanced_accuracy_score(y_test, test_prediction),
        "Specificity": recall_score(y_test, test_prediction, pos_label=0)})
test_comparison = pd.DataFrame(test_scores).set_index("Model")
display((test_comparison * 100).style.format("{:.2f}%"))


,Accuracy,Precision,Diabetes recall,F1-score,AUROC,PR-AUC,Balanced accuracy,Specificity
Model,,,,,,,,
Gradient Boosting,76.31%,41.04%,69.27%,51.54%,79.11%,43.14%,73.57%,77.87%
CatBoost,73.82%,38.69%,75.12%,51.08%,79.73%,44.81%,74.33%,73.54%


In [39]:
test_difference = (test_comparison.loc["CatBoost"] - test_comparison.loc["Gradient Boosting"]) * 100
display(test_difference.to_frame("CatBoost - Gradient Boosting (percentage points)").round(2))

,CatBoost - Gradient Boosting (percentage points)
Accuracy,-2.48
Precision,-2.35
Diabetes recall,5.85
F1-score,-0.46
AUROC,0.62
PR-AUC,1.67
Balanced accuracy,0.76
Specificity,-4.34
